# 02. EDA

Price history, log returns, distributions, volatility, cross-series
relationships, external-series behavior, and the ADRO/AADI event window.
Descriptive only. No causal claims.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from coal_forecasting.config import load_config
from coal_forecasting.data import load_snapshots

config = load_config(Path('configs/baseline.toml'))
snaps = load_snapshots(config)

In [ ]:
for symbol in ('ADRO.JK', 'PTBA.JK', 'ITMG.JK', 'IDR=X'):
    f = snaps[symbol].frame.sort_values('Date')
    r = np.log(f['Adj Close']).diff()
    print(f"{symbol}: mean={r.mean():.6f} std={r.std():.6f} "
          f"min={r.min():.4f} max={r.max():.4f} skew={r.skew():.3f} excess_kurt={r.kurt():.3f}")

In [ ]:
from scipy import stats

for symbol in ("ADRO.JK", "PTBA.JK", "ITMG.JK"):
    f = snaps[symbol].frame.sort_values("Date")
    r = np.log(f["Adj Close"]).diff().dropna().to_numpy(float)
    jb, p_value = stats.jarque_bera(r)
    print(f"{symbol}: excess_kurt={stats.kurtosis(r):.3f} jb={jb:.1f} p={p_value:.2e}")
print("p far below any level: normality rejected for all three.")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, symbol in zip(axes, ('ADRO.JK', 'PTBA.JK', 'ITMG.JK')):
    f = snaps[symbol].frame.sort_values('Date')
    ax.plot(f['Date'], f['Adj Close'])
    ax.set_title(f'{symbol} adjusted close')
    if symbol == 'ADRO.JK':
        ax.axvspan(pd.Timestamp('2024-10-01'), pd.Timestamp('2025-03-31'),
                   alpha=0.2, label='ADRO/AADI restructuring window')
        ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Lagged correlation of each external return with the ADRO target return.
from dataclasses import replace
from coal_forecasting.features import build_feature_dataset

fc = replace(config.features, group='E3')
d, cols = build_feature_dataset(snaps, 'ADRO.JK', fc)
for lag in (1, 2, 3, 5, 10):
    c = d['target_next_day_log_return'].corr(d[f'own_return_lag_{lag}'])
    fx = d['target_next_day_log_return'].corr(d[f'usd_idr_return_lag_{lag}'])
    print(f'lag {lag}: own={c:+.4f} usd_idr={fx:+.4f}')
print('Weak lagged correlations are expected; correlation is not causation.')